In [ ]:
import os
from huggingface_hub import hf_hub_download
import duckdb

# download
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

con = duckdb.connect()

# Aggregate to page-level: first half vs second half of March
df = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_avg_position,
            gsc_clicks,
            CASE WHEN report_date <= DATE '2026-03-15' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT
            content_hash_id,
            client_hash_id,
            period,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM daily
        GROUP BY 1,2,3
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.impressions AS impressions_first,
        s.impressions AS impressions_second,
        f.avg_position AS avg_position_first,
        s.avg_position AS avg_position_second
    FROM agg f
    JOIN agg s
      ON f.content_hash_id = s.content_hash_id
     AND f.client_hash_id = s.client_hash_id
    WHERE f.period = 'first_half' AND s.period = 'second_half'
""").df()

df.shape

(141467, 6)

In [3]:
import pandas as pd

df["pct_change_impressions"] = (df["impressions_second"] - df["impressions_first"]) / df["impressions_first"].replace(0, pd.NA)
df["is_declining"] = (df["pct_change_impressions"] < -0.10).astype(int)  # >10% drop = declining

df["is_declining"].value_counts(normalize=True)

is_declining
0    0.660416
1    0.339584
Name: proportion, dtype: float64

In [4]:
df["position_bucket"] = pd.cut(
    df["avg_position_first"],
    bins=[0, 10, 20, 30, 50, 1000],
    labels=["1-10", "11-20", "21-30", "31-50", "50+"]
)

position_check = df.groupby("position_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
).reset_index()

position_check

/var/folders/3y/dct87j3j2yl68z27_6ws2wpr0000gn/T/ipykernel_84219/659279335.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  position_check = df.groupby("position_bucket").agg(


,position_bucket,n,decline_rate
0,1-10,78973,0.353032
1,11-20,25894,0.325249
2,21-30,13400,0.347836
3,31-50,13347,0.349667
4,50+,8985,0.258765


In [5]:
df["position_bucket"] = pd.cut(
    df["avg_position_first"],
    bins=[0, 10, 20, 30, 50, 1000],
    labels=["1-10", "11-20", "21-30", "31-50", "50+"]
)

position_check = df.groupby("position_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
).reset_index()

position_check

/var/folders/3y/dct87j3j2yl68z27_6ws2wpr0000gn/T/ipykernel_84219/659279335.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  position_check = df.groupby("position_bucket").agg(


,position_bucket,n,decline_rate
0,1-10,78973,0.353032
1,11-20,25894,0.325249
2,21-30,13400,0.347836
3,31-50,13347,0.349667
4,50+,8985,0.258765


In [6]:
df["impressions_bucket"] = pd.cut(
    df["impressions_first"],
    bins=[0, 10, 100, 500, 2000, 1000000],
    labels=["0-10", "11-100", "101-500", "501-2000", "2000+"]
)

impressions_check = df.groupby("impressions_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
).reset_index()

impressions_check

/var/folders/3y/dct87j3j2yl68z27_6ws2wpr0000gn/T/ipykernel_84219/181695817.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  impressions_check = df.groupby("impressions_bucket").agg(


,impressions_bucket,n,decline_rate
0,0-10,23670,0.244740
1,11-100,40643,0.348621
2,101-500,35398,0.363043
3,501-2000,26404,0.353621
4,2000+,15352,0.383663


Section 1: Signal Checks

- `avg_position`: Bucket table above. Decline rate is flat (~33-35%) across positions1-50, dropping to ~26% only at position 50+ so this isn't clean, monotonic relationship between position and decline in this window.

- Verdict: MIXED because Position alone doesn't reliably separate declining from stable pages here, consistent with the surprising, non-monotonic pattern that I found in Week 1. I think this is a useful negative because rule shouldn't lean heavily on position alone as a decline signal.

- `gsc_impressions`: Bucket table above. Decline rate rises steadily and monotonically from 24.5% (0-10 impressions) to 38.4% (2000+ impressions).

Verdict: CONFIRMED but in the opposite direction from a naive "quick-win" assumption. High-volume pages decline more often so this make volume a genuinely useful signal

In [7]:
import numpy as np

# Score: normalize impressions_first to 0-100 scale
df["baseline_score"] = (
    100 * (df["impressions_first"] - df["impressions_first"].min())
    / (df["impressions_first"].max() - df["impressions_first"].min())
)

# ONE reason code — this will assign when the page is actually high-volume
df["reason_code"] = np.where(
    df["impressions_first"] >= 500,
    "high_visibility_risk",
    "low_visibility"
)

# Action label — simple threshold on the score
threshold = df["baseline_score"].quantile(0.90)  # top 10% flagged for review
df["action"] = np.where(df["baseline_score"] >= threshold, "review", "monitor")

df[["content_hash_id", "client_hash_id", "impressions_first", "baseline_score", "reason_code", "action"]].sort_values(
    "baseline_score", ascending=False
).head(10)

,content_hash_id,client_hash_id,impressions_first,baseline_score,reason_code,action
39723,content_eadb33b5df496f4a,client_e547b89c05043229,161575.0,100.000000,high_visibility_risk,review
28089,content_e8a52cf3d5988c07,client_23a62021009f63c4,143173.0,88.610791,high_visibility_risk,review
67539,content_ec2e0346994fb5a5,client_e547b89c05043229,132811.0,82.197631,high_visibility_risk,review
74786,content_36e53e9c707674fc,client_23a62021009f63c4,109909.0,68.023321,high_visibility_risk,review
97670,content_7172a7fad43f0998,client_62f4a7e64f5e0096,108663.0,67.252157,high_visibility_risk,review
61490,content_b99ea6861864dea5,client_62f4a7e64f5e0096,91474.0,56.613688,high_visibility_risk,review
130170,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,87662.0,54.254397,high_visibility_risk,review
97665,content_7c6373141eae744a,client_62f4a7e64f5e0096,86860.0,53.758030,high_visibility_risk,review
31465,content_3df3f32f3fd58dea,client_23a62021009f63c4,84041.0,52.013319,high_visibility_risk,review
114596,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83772.0,51.846832,high_visibility_risk,review


In [8]:
import os

os.makedirs("work/outputs", exist_ok=True)

output = df[[
    "content_hash_id", "client_hash_id", "impressions_first",
    "avg_position_first", "baseline_score", "reason_code", "action"
]].sort_values("baseline_score", ascending=False)

output.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(output.shape)

(141467, 7)


Section 2:

Score: `baseline_score`
- first-half March impressions (`impressions_first`), min-max normalized to 0-100. Uses only pre-decision-point data, nothing from the second half of the month or the decline label itself.

Reason code: `high_visibility_risk`
- Assigned when `impressions_first >= 500`
- `low_visibility` assigned when <= 500
=> Reflecting signal 1 finding are high-volume pages decline more often, so visibility 
itself is the risk driver here, not position.

Action label: `review` for the top 10% of scores, `monitor` for 
everything else.
=> Simple two-tier split matching a reviewer's limited capacity.

Why this rule: 
- Signal 2(impressions) shows a clear, monotonic relationship with decline from 24.5% → 38.4% across 
buckets.
- Signal 1(position) shows no such pattern

=> The rule is built on the signal that actually held up, not the one I originally expected 
to matter.

Output written to `work/outputs/baseline_action_score.csv` (141,467 
rows, ranked by `baseline_score` descending).


Section 3: Top-10 review

1. content_eadb33b5df496f4a (161,575 impressions): For review as the single highest-volume page in the sample and it would be wrong if this page is a stable evergreen top-performer with no actual movement and high volume alone doesn't confirm decline is happening.

2. content_e8a52cf3d5988c07 (143,173 impressions): Same client-tier reasoning and it would be wrong if this page traffic is seasonal and naturally fluctuates, not declining.

3. content_ec2e0346994fb5a5 (132,811 impressions):  Same client as (client_e547b89c05043229). Two of this client's pages both in the top 3 and would be wrong if this is a single client's traffic pattern 
dominating the ranking rather than a genuinely at-risk page.

4. content_36e53e9c707674fc (109,909 impressions): Flag purely on voulume and it would be wrong if he page's position and clicks are already strong and stable because volume alone says nothing about direction.

5. content_7172a7fad43f0998 (108,663 impressions):  Same client as #6 and #8 (client_62f4a7e64f5e0096). Three of these client's pages in the top 10 and it would be wrong if reflecting one client's overall size rather than page-level risk so the rule may just be ranking big 
clients' pages, not risky ones.

6. content_b99ea6861864dea5 (91,474 impressions): Same concern as #5: client concentration risk in the ranking.

7. content_e7b5dd4dff461ad2 (87,662 impressions): It would be wrong if this page is a recently launched due to still-growing page misclassified as "at risk" simply because of size.

8. content_7c6373141eae744a (86,860 impressions): Same client concentration concern as #5 and #6.

9. content_3df3f32f3fd58dea (84,041 impressions): It would be wrong if this page has been consistently high-volume for a long time with no sign of decline and i think a volume without a trend signal doesn't prove risk.

10. content_9c057b66c30a3abb (83,772 impressions): It would be wrong for the same reason: no direct evidence of movement, only size.

Section 4: Weak Picks

The clearest weakness in this top 10 is:
- client concentration: client_62f4a7e64f5e0096 alone accounts for 3 of the top 10 rows, (#5, #6, #8), and client_e547b89c05043229 accounts for 2 (#1, #3).

=> Suggesting the rule may be partly ranking "which clients have the biggest pages" rather than "which specific pages are actually at risk and a stronger version of this rule would normalize impressions within each client, or cap how many pages from one client can appear in the top list, so no single large client dominates the queue.



Section 5: Self-Check

- Two signal verdicts with bucket tables + n: done (position = MIXED, impressions = CONFIRMED; impressions ties to the "quick-win/volume" flag).
- One rule with score, one reason code, one action label: done (baseline_score, high_visibility_risk, review/monitor).
- Ranked queue written from the notebook: done (work/outputs/baseline_action_score.csv, 141,467 rows).
- Ten reviewed rows with "what would make it wrong": done.
- No future-window or label-derived inputs: confirmed and the rule only uses impressions_first, never impressions_second or is_declining.